# BDB27 — is the 30–60° DB turn important?

**Two questions, asked separately.**

1. **What the turn costs.** Across all 58,664 valid DB game turns, is a 30–60° turn a physically
   demanding movement or a gentle one, and where does the band sit on the cost curve?
2. **What the turn is worth.** Are plays in which a DB has to turn 30–60° worse for the defence
   than plays in which he does not turn — and how does that band compare with sharper turns?

**This is not a separation test.** `separation_at_pass_forward` is never read. There is no receiver
matching, no attribution inference and no proximity rule. Every Test 2 row is a DB's **own**
`player_play` row, joined on `(game_id, play_id, nfl_id)`.

**Test 2 is stakes, not cause.** A turn exists because the route broke. The estimates say what is
at stake when a turn happens, conditional on controls; they do not say that turning causes the
outcome. Every Test 2 number carries the label `ASSOCIATION (STAKES), NOT CAUSE`.

**Spec.** `notes/turn-importance-spec.md`, committed before any result. **Decision:** D223.

Press **Run All**. Nothing else to edit except §1 if your files live elsewhere.

## 1. Paths — the only cell to edit

* `TURNS_FULL` — the FULL DB game turn table. `None` means
  `data/interim/full_run_20261111/bdb27_full_db_turns.parquet`.
* `PLAYER_PLAY` — `data/interim/parquet/player_play.parquet`.
* `GAMES` — `data/interim/parquet/games.parquet` (season lookup only).
* `TURNS_SAMPLE` — `outputs/turns/turns.parquet`; declared in the provenance block and **not
  used** for any estimate (it is the 2023 sample).
* `OUT_DIR` — where the CSV/JSON artefacts are written. `None` means `outputs/turn_importance`.

The notebook reads Parquet only. It does not need the raw CSVs.

In [ ]:
# ======================= CONFIG — the only cell to edit =========================
TURNS_FULL   = None   # None = data/interim/full_run_20261111/bdb27_full_db_turns.parquet
PLAYER_PLAY  = None   # None = data/interim/parquet/player_play.parquet
GAMES        = None   # None = data/interim/parquet/games.parquet
TURNS_SAMPLE = None   # None = outputs/turns/turns.parquet  (provenance only, unused)
OUT_DIR      = None   # None = outputs/turn_importance
# =============================================================================

## 2. Setup

Walks up to the repository that holds `config.yaml` and `src/bdb27`, puts `src/` on the path and
imports `bdb27.turn_importance` — the same module the tests exercise, so the notebook cannot drift
from the tested code path.

In [ ]:
import importlib
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "config.yaml").exists() and (parent / "src" / "bdb27").is_dir():
            return parent
    raise RuntimeError("repository root not found: no parent holds config.yaml and src/bdb27")


REPO = find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import bdb27.turn_importance as ti          # noqa: E402
importlib.reload(ti)

print("repo:", REPO)
print("module:", Path(ti.__file__).resolve())
print("label:", ti.LABEL)

## 3. Inputs

Every path is resolved, printed with its size and SHA-256, and the required columns are checked
before anything runs. The only conditions that stop the notebook are a missing file, an unreadable
file or a missing required column — and they name every path tried. Small n, a thin band or a null
result never stop it.

In [ ]:
paths = {
    "turns_full":   Path(TURNS_FULL) if TURNS_FULL else REPO / "data/interim/full_run_20261111/bdb27_full_db_turns.parquet",
    "player_play":  Path(PLAYER_PLAY) if PLAYER_PLAY else REPO / "data/interim/parquet/player_play.parquet",
    "games":        Path(GAMES) if GAMES else REPO / "data/interim/parquet/games.parquet",
    "turns_sample": Path(TURNS_SAMPLE) if TURNS_SAMPLE else REPO / "outputs/turns/turns.parquet",
}
for role, path in paths.items():
    print(f"{role:<13} {path}")
    print(f"{'':<13} exists={path.exists()} bytes={path.stat().st_size if path.exists() else None}")

out_dir = Path(OUT_DIR) if OUT_DIR else REPO / "outputs/turn_importance"
print("\noutputs ->", out_dir)

## 4. Run both tests

One call runs Test 1 and Test 2, prints the log and writes every artefact to `OUT_DIR`:
band tables, the cost tables, the counts, every specification, the gradient and `summary.json`.
All numbers below are printed by that call, with their denominators.

In [ ]:
RESULTS = ti.run(
    turns_full=paths["turns_full"],
    pp_path=paths["player_play"],
    games_path=paths["games"],
    out_dir=out_dir,
    turns_sample=paths["turns_sample"],
)

## 5. Test 1 in one table

`1` is the gentlest band on that measure. Direction: higher peak turn rate and higher peak lateral
acceleration are costlier; lower speed retention, lower apex retention and smaller path radius are
costlier. Entry, apex and exit speed are context, not cost.

In [ ]:
rank_pivot = RESULTS["wide"][["band", "n_turns", "share_of_turns",
                              "entry_speed_median", "apex_speed_median",
                              "exit_speed_median", "speed_retention_median",
                              "apex_retention_median", "peak_omega_median",
                              "peak_lat_accel_median", "path_radius_median",
                              "plant_share"]].copy()
rank_pivot["share_of_turns"] = (rank_pivot["share_of_turns"] * 100).round(2)
rank_pivot["plant_share"] = (rank_pivot["plant_share"] * 100).round(1)
print(rank_pivot.round(2).to_string(index=False))

## 6. Test 2 in one table

Coefficients are relative to **no turn** on the same play, on pass plays, with player fixed
effects and down, distance, coverage, position and season controls; 95% CI two-way clustered by
player and game. Positive = worse for the defence.

In [ ]:
import pandas as pd  # noqa: E402

est = RESULTS["estimates"]
show = est[est["spec"].isin(["primary_epa", "primary_pass_success",
                             "primary_yards", "primary_explosive"])]
print(show[["spec", "band", "coef", "se", "ci_lo", "ci_hi", "n_rows",
            "n_plays", "n_players"]].round(4).to_string(index=False))

## 7. The pre-declared verdict labels

These are the labels fixed in the spec before the model was fitted. They describe the estimate;
none of them upgrades an association to a cause.

In [ ]:
print("Test 1 — 30–60° on the physical cost curve:")
print("  ", RESULTS["summary"]["test1"]["severity_label"],
      "| ranks:", RESULTS["summary"]["test1"]["ranks"])

print("\nTest 2 — 30–60° vs no turn (pass plays):")
for spec, lab in RESULTS["labels"].items():
    print(f"  {spec:<22} beta={lab['coef']:+.4f}  95% CI "
          f"[{lab['ci_lo']:+.4f}, {lab['ci_hi']:+.4f}]  n={lab['n']:,}  "
          f"players={lab['players']:,}  -> {lab['decision']}")

grad = RESULTS["gradient"]
bigger = grad[grad["ci_lo"] > 0]
print(f"\nSharper bands whose EPA contrast exceeds 30–60° with a CI excluding 0: "
      f"{len(bigger)} of {len(grad)}")
print(grad[["band", "coef", "ci_lo", "ci_hi"]].round(4).to_string(index=False))
print("\nlabel for every estimate:", ti.LABEL)

## 8. What this does and does not say

* **It does say** where 30–60° sits on the physical cost curve, and what is at stake on plays
  where a DB turns that far, with uncertainty and with the sharper bands beside it.
* **It does not say** that turning 30–60° causes anything. A turn is post-snap and endogenous: the
  route broke, so the DB turned. Test 2 measures stakes.
* **A null is not "unimportant".** Non-detection is reported as non-detection, never as evidence
  that the band does not matter.
* **Frequency is not severity.** 30–60° is 13.3% of valid DB game turns (7,777 / 58,664) and is the
  mildest band the detector records (the 30° floor is applied upstream, so no turn in the file
  falls below it). Whatever the cost curve shows, the Combine's shortfall in this band is a
  frequency fact — 6.01× under-represented by share in `TASK.md` — not a claim that 30–60° is the
  most punishing turn.